# Module 4 · Multi-agent: Triage → Loan Service handoff
⏱ ~10–15 min, then the bridge into the Module 5 capstone

**In this module**
1. Why split one agent into several
2. Least privilege: each agent sees only its own tools
3. Routing with a Triage Agent
4. The handoff loop, written by hand (~35 lines)
5. Context carries over, including a hand *back* to triage
6. Lesson from rehearsal: a router with too many tools
7. Alternative pattern: an **orchestrator** that calls specialists as tools

### Why split?
| Single agent | Multi-agent (triage + specialists) |
|---|---|
| One prompt that grows with every domain (loans, KYC, cards…) | Small, focused prompt per domain |
| Every tool reachable from every conversation | **Least privilege**: each agent sees only its own tools |
| One shared component to test and approve | Each business team owns and tests its own agent |
| Fewer LLM calls | One extra routing call per conversation |

```
                ┌──────────────────┐   transfer_to_loan_service   ┌────────────────────┐
 customer ────► │   Triage Agent   │ ───────────────────────────► │ Loan Service Agent │
                └──────────────────┘ ◄─────────────────────────── └────────────────────┘
                   │ FAQ → answers itself    transfer_to_triage        (future: KYC, Fraud,
                   │ out of scope → helpline                            Collections agents)
```
> 🗣️ *Speaker cue:* the key idea is that **a handoff is just a tool call**. When an agent calls `transfer_to_X`, our code swaps the system prompt and toolset. The conversation history stays.

In [1]:
import os, sys, json
sys.path.append("..")                      # so we can import utils.py and harness/

from dotenv import load_dotenv
import aisuite as ai
from utils import print_html, print_json, show_messages, show_tool_call, show_trace, show_usage, compare

load_dotenv("../.env", override=True)
MODEL = os.getenv("MODEL", "openai:gpt-4o-mini")   # aisuite format -> "provider:model"
client = ai.Client()                               # reads OPENAI_API_KEY from the environment

print_html({"model": MODEL, "client": "aisuite.Client() ready"}, title="Setup", color="gray")

model,openai:gpt-4o-mini
client,aisuite.Client() ready


## Step 1 · Define two agents with least privilege
Each agent is just **a system prompt plus a list of tools**. The handoff functions do no business work; they only return where to go next.

In [2]:
from aisuite.utils.tools import Tools
from harness.tools import check_loan_status, get_missing_documents, get_document_checklist, escalate_to_human

TRIAGE_PROMPT = """You are the Triage Agent for Demo Bank's customer chat. You only route.
- Anything about a loan application (status, documents, decisions, complaints): call transfer_to_loan_service.
- Simple general banking questions: answer briefly yourself. Branch hours: 9:30-16:00, Mon-Sat.
- Anything else (investments, cards, fraud): no advice. Point to the 24x7 helpline 1800-000-0000 or a branch."""

LOAN_PROMPT = """You are the Loan Service Agent for Demo Bank.
- Use your tools for any fact about an application. Never guess.
- You cannot approve, reject or change loans.
- Escalate complaints with escalate_to_human (priority high) and share the ticket ID.
- Be concise and warm."""

def transfer_to_loan_service(reason: str) -> dict:
    """Hand the conversation to the Loan Service Agent. Use for anything about a loan application.

    Args:
        reason: One-line summary of what the customer needs.
    """
    return {"transfer_to": "loan_service", "reason": reason}

def transfer_to_triage(reason: str) -> dict:
    """Hand the conversation back to the Triage Agent when the customer asks about something
    that is NOT a loan application (e.g. branch hours, cards, investments).

    Args:
        reason: One-line summary of the new request.
    """
    return {"transfer_to": "triage", "reason": reason}

AGENTS = {
    "triage": {"prompt": TRIAGE_PROMPT,
               "tools": [transfer_to_loan_service]},
    "loan_service": {"prompt": LOAN_PROMPT + "\n- If the customer asks about something else, call transfer_to_triage.",
                     "tools": [check_loan_status, get_missing_documents, escalate_to_human, transfer_to_triage]},
}

print_html([{"agent": name, "can call": ", ".join(f.__name__ for f in a["tools"])} for name, a in AGENTS.items()],
           title="Least privilege: each agent sees only its own tools", color="teal")

agent,can call
triage,transfer_to_loan_service
loan_service,"check_loan_status, get_missing_documents, escalate_to_human, transfer_to_triage"


## Step 2 · The Triage Agent's routing decisions
One LLM call per message. Either the model calls `transfer_to_loan_service`, or it answers itself.

In [3]:
def route(message: str) -> dict:
    agent = AGENTS["triage"]
    r = client.chat.completions.create(
        model=MODEL, temperature=0, tools=Tools(agent["tools"]).tools(),
        messages=[{"role": "system", "content": agent["prompt"]}, {"role": "user", "content": message}])
    m = r.choices[0].message
    if m.tool_calls:
        args = json.loads(m.tool_calls[0].function.arguments)
        return {"message": message, "decision": "→ loan_service", "detail": args.get("reason")}
    return {"message": message, "decision": "answered by triage", "detail": m.content}

print_html([route(q) for q in [
    "What's the status of LA-1001?",
    "Which documents are missing for LA-1002?",
    "LA-1004 was rejected without any reason. I want to file a complaint!",
    "What are your branch timings?",
    "Should I invest in mutual funds?",
]], title="Triage routing decisions", color="teal")

message,decision,detail
What's the status of LA-1001?,→ loan_service,Status inquiry for loan application LA-1001.
Which documents are missing for LA-1002?,→ loan_service,Inquiry about missing documents for loan application LA-1002.
LA-1004 was rejected without any reason. I want to file a complaint!,→ loan_service,Filing a complaint regarding loan application LA-1004 rejection.
What are your branch timings?,answered by triage,"Our branch hours are 9:30 AM to 4:00 PM, Monday to Saturday."
Should I invest in mutual funds?,answered by triage,I can't provide advice on investments. I recommend contacting our 24x7 helpline at 1800-000-0000 or visiting a branch for assistance.


## Step 3 · The handoff loop, by hand
This is the single-agent loop from Module 3 with **one change**: the active agent can switch. When the model calls a `transfer_to_*` tool, we change `active`. The next LLM call then uses the new agent's prompt and tools, with the **same history**.

In [4]:
def run_conversation(user_messages: list[str], start: str = "triage", max_steps: int = 6):
    history, active, trace = [], start, []
    for user_text in user_messages:
        history.append({"role": "user", "content": user_text})
        trace.append({"type": "user", "agent": active, "data": user_text})

        for _ in range(max_steps):                                  # step limit also stops ping-pong
            agent = AGENTS[active]
            registry = {f.__name__: f for f in agent["tools"]}       # this agent's allow-list
            r = client.chat.completions.create(
                model=MODEL, temperature=0, tools=Tools(agent["tools"]).tools(),
                messages=[{"role": "system", "content": agent["prompt"]}, *history])
            m = r.choices[0].message

            if not m.tool_calls:                                     # this agent is done
                history.append({"role": "assistant", "content": m.content})
                trace.append({"type": "final", "agent": active, "data": m.content})
                break

            history.append({"role": "assistant", "content": m.content, "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in m.tool_calls]})
            for tc in m.tool_calls:
                name, args = tc.function.name, json.loads(tc.function.arguments or "{}")
                result = registry[name](**args) if name in registry else {"error": f"{name} not approved"}
                if name.startswith("transfer_to_"):                  # ← THE HANDOFF
                    trace.append({"type": "handoff", "agent": active,
                                  "data": f"{active} → {result['transfer_to']} · {args.get('reason')}"})
                    active = result["transfer_to"]
                else:
                    trace.append({"type": "escalation" if name == "escalate_to_human" else "tool_call",
                                  "agent": active, "data": {"tool": name, "args": args, "result": result}})
                history.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    return history, trace

## Step 4 · A handoff in action

In [5]:
history, trace = run_conversation(["Hi, where does my application LA-1002 stand?"])
show_trace(trace, title="Triage → Loan Service")
print_html(history[-1]["content"], title="Customer sees", color="green")

## Step 5 · Context carries over, and the hand-back
Two customer messages in one conversation. The second is not about a loan, so the Loan Service Agent hands control **back** to triage. Nobody asks the customer to repeat anything.

In [6]:
history, trace = run_conversation([
    "What's the status of LA-1001?",
    "Thanks. Also, what time does the branch open on Saturday?",
])
show_trace(trace, title="Triage → Loan Service → Triage")
show_messages(history, title="Shared message history (both agents read the same history)", show_system=False)

## Step 6 · Lesson from rehearsal: a router with too much power
In an early version, the Triage Agent also had `escalate_to_human`. On a complaint, it raised the ticket **itself**, so the Loan Service Agent and its complaint playbook never ran.
Let's re-run that experiment. LLM behaviour varies, so the "too much power" run may or may not short-circuit this time. The point is that it **can**.

In [7]:
complaint = "LA-1004 was rejected without any reason. I want to file a complaint!"

def steps(trace):
    return "\n".join(f"- **{e['agent']}** · {e['type'].replace('_', ' ')}" for e in trace if e["type"] != "user")

AGENTS["triage"]["tools"] = [transfer_to_loan_service, escalate_to_human]   # ❌ router can act
_, trace_bad = run_conversation([complaint])

AGENTS["triage"]["tools"] = [transfer_to_loan_service]                      # ✅ router only routes
_, trace_good = run_conversation([complaint])

compare(steps(trace_bad), steps(trace_good),
        "Triage WITH escalate_to_human", "Triage routes only (least privilege)")

> 🗣️ *Speaker cue:* least privilege applies to routing agents too. If the router can act, it will sometimes act instead of routing.

## Step 7 · Alternative pattern: an orchestrator (agents as tools)
With a **handoff**, control *moves* to the specialist. With an **orchestrator**, a manager agent *calls* specialists as tools and combines their answers. That fits requests that span more than one area.

Here each specialist is itself a small agent, using aisuite's `max_turns` auto-loop from Module 2.

In [8]:
def ask_loan_specialist(question: str) -> str:
    """Ask the Loan Service specialist about a specific customer's loan application
    (status, stage, missing documents). Include the application ID in the question.

    Args:
        question: Self-contained question including the application ID.
    """
    r = client.chat.completions.create(model=MODEL, max_turns=4, tools=[check_loan_status, get_missing_documents],
        messages=[{"role": "system", "content": LOAN_PROMPT}, {"role": "user", "content": question}])
    return r.choices[0].message.content

def ask_product_specialist(question: str) -> str:
    """Ask the Loan Products specialist general questions about loan products and the
    documents required to apply for a type of loan.

    Args:
        question: Self-contained product question.
    """
    r = client.chat.completions.create(model=MODEL, max_turns=4, tools=[get_document_checklist],
        messages=[{"role": "system", "content": "You answer questions about Demo Bank loan products. "
                                                "Use get_document_checklist for document requirements."},
                  {"role": "user", "content": question}])
    return r.choices[0].message.content

ORCHESTRATOR_PROMPT = """You are the coordinator for Demo Bank's loan desk.
Break the customer's message into parts, ask the right specialist for each part,
then write ONE short, combined reply. Never invent facts the specialists did not give you."""

question = ("My personal loan LA-1002 is in progress. Where does it stand? "
            "Also, which documents would I need if I apply for a car loan?")
result = client.chat.completions.create(
    model=MODEL, max_turns=5, tools=[ask_loan_specialist, ask_product_specialist],
    messages=[{"role": "system", "content": ORCHESTRATOR_PROMPT}, {"role": "user", "content": question}])

show_messages(result.choices[0].intermediate_messages, title="Orchestrator calling specialists as tools")
print_html(result.choices[0].message.content, title="Combined answer", color="green")

### Handoff vs orchestrator
| | Handoff (used in our harness) | Orchestrator |
|---|---|---|
| Who talks to the customer | The specialist, directly | The orchestrator only |
| Best for | One domain per request | Requests spanning several domains |
| Cost | One extra routing call | Extra calls per specialist, plus a final composition |
| Audit trail | A clean chain of handoffs | Nested calls, so log each specialist call |

## ✅ Module 4 takeaways
- A **handoff is just a tool call**; the loop swaps the prompt and toolset and keeps the history.
- **Least privilege** for every agent, including the router.
- Start with one agent. Split when domains, owners or tool permissions diverge.
- Handoff and orchestrator are both valid patterns; choose by request shape.

---
# Module 5 · Capstone: the BFSI Agent Harness
Everything so far, packaged into `harness/` with governance built in:

| Governance rule (from the agenda) | How the harness enforces it |
|---|---|
| Use only approved tools | Per-agent **allow-list**. Unknown tool calls are **blocked** and logged |
| Agents must not approve or reject loans | No such tool + decision-request detector + **output check** that replaces any "I approved…" reply |
| Protect customer data | **PII masking** (PAN, Aadhaar, card, phone, email, account no.) *before* the LLM sees the text |
| Maintain human escalation | `escalate_to_human` tool + automatic escalation when the step limit is hit |
| Record agent actions for audit | Every step → `logs/audit.jsonl` |

**Skills** = Markdown playbooks in `harness/skills/*/SKILL.md`. Only the name and description sit in the prompt. The agent calls `load_skill` to pull the full playbook when needed. Business or compliance teams can review and change behaviour **without touching code**.

In [9]:
from harness import build_default_harness

h = build_default_harness()
print_html([{"skill": s.name, "description": s.description} for s in h.skills.values()],
           title="Skills discovered in harness/skills/", color="purple")

skill,description
complaint-handling,"Playbook for complaints, frustration, delays, or requests to speak to a person. Load whenever the customer sounds unhappy or explicitly asks for a human."
decision-requests,"What to do when a customer asks the assistant to approve, reject, sanction, speed up, waive fees on, or change the terms of a loan. Load for any request to change a loan decision."
loan-status-explainer,"How to explain each loan status code to a customer in plain language, including what happens next and timelines. Load before answering any loan status question."
missing-documents,"Guidance for telling customers which documents are missing or required, accepted formats, and how to upload them. Load for any document-related question."


## Tools vs skills: what's the difference?
People often mix these up, because the agent *loads* a skill through a tool call (`load_skill`). The difference is what comes back and what it is used for.

| | 🔧 **Tool** | 📘 **Skill** |
|---|---|---|
| **What it is** | A Python function (code) | A Markdown playbook (text) |
| **What it gives the agent** | **Data or an action**: a loan's status, a new ticket | **Know-how**: *how* to handle a type of request |
| **Who runs it** | Our application **executes** it | Nothing is executed; the text is added to the model's context |
| **Side effects** | Can have them (`escalate_to_human` creates a ticket) | None |
| **How the model discovers it** | Its JSON schema is sent in `tools=` on every call | Only its name and one-line description sit in the system prompt; full text loaded on demand |
| **Who writes and changes it** | Developers, with code review and tests | Business or compliance experts, with content review |
| **Analogy** | The bank systems a staff member is allowed to use | The SOP manual on their desk |

**In one line:** the skill says *what to do*; the tool *does it*. For a complaint, the `complaint-handling` skill says "acknowledge, check facts, escalate with high priority, confirm the ticket", and the `escalate_to_human` tool actually raises the ticket.

In [10]:
from harness.skills import make_load_skill_tool

tool_output  = check_loan_status("LA-1002")                                  # runs code -> DATA
skill_output = make_load_skill_tool(h.skills)("missing-documents")           # reads text -> INSTRUCTIONS

compare(tool_output, skill_output["instructions"],
        "🔧 Tool · check_loan_status(\"LA-1002\") → data",
        "📘 Skill · missing-documents → instructions",
        left_color="amber", right_color="purple")

found,True
application_id,LA-1002
loan_type,Personal Loan
status,DOCUMENTS_PENDING
stage,Document verification
submitted_on,2026-11-05
last_updated,2026-11-10
expected_decision_by,None
has_missing_documents,True
officer_note,Cannot proceed until income documents are uploaded.


### What the model sees up front: full tool schemas, but only a short catalogue of skills

In [11]:
loan_agent = h.agents["loan_service"]
schemas = Tools(list(h._registry(loan_agent).values())).tools()
print_html([{"tool": s["function"]["name"], "parameters": ", ".join(s["function"]["parameters"].get("properties", {}))}
            for s in schemas], title="🔧 Tool schemas sent on every call (tools=...)", color="amber")
prompt = h._system_prompt(loan_agent)
print_html(prompt[prompt.index("AVAILABLE SKILLS"):], title="📘 Skill catalogue in the system prompt (full text loaded only when needed)", color="purple")

tool,parameters
check_loan_status,application_id
get_missing_documents,application_id
get_document_checklist,loan_type
escalate_to_human,"application_id, reason, priority"
load_skill,skill_name
transfer_to_triage,reason


In [12]:
def run(message: str):
    result = h.chat(message)
    print_html(result.answer, title=f"🧑 {message}", subtitle=f"answered by {result.agent}", color="green")
    show_trace(result.events, title="Harness trace")
    h.reset()

run("Hi, what is the status of my loan application LA-1001?")

### PII is masked *before* the model sees it

In [13]:
run("My PAN is ABCDE1234F and mobile 9876543210. Can you check LA-1005?")

### Decision request → policy reminder + `decision-requests` skill

In [14]:
run("Please approve my loan LA-1001 right now.")

### Complaint → `complaint-handling` skill + high-priority escalation

In [15]:
run("LA-1004 was rejected without any reason. I want to file a complaint!")

### The audit log

In [16]:
audit = h.audit.read()
print_html([{"ts": r["ts"][11:19], "agent": r["agent"], "event": r["type"],
             "data": json.dumps(r["data"], default=str)[:110]} for r in audit],
           title=f"logs/audit.jsonl · session {h.session_id}", color="gray")

ts,agent,event,data
15:04:32,triage,user,"""Hi, what is the status of my loan application LA-1001?"""
15:04:34,triage,llm_call,"{""step"": 1, ""model"": ""openai:gpt-4o-mini"", ""tool_calls"": [""transfer_to_loan_service""], ""tokens"": 291}"
15:04:34,triage,handoff,"{""from"": ""triage"", ""to"": ""loan_service"", ""reason"": ""Customer inquiring about the status of loan application LA"
15:04:35,loan_service,llm_call,"{""step"": 2, ""model"": ""openai:gpt-4o-mini"", ""tool_calls"": [""load_skill""], ""tokens"": 893}"
15:04:35,loan_service,skill_loaded,"""loan-status-explainer"""
15:04:37,loan_service,llm_call,"{""step"": 3, ""model"": ""openai:gpt-4o-mini"", ""tool_calls"": [""check_loan_status""], ""tokens"": 1274}"
15:04:37,loan_service,tool_call,"{""name"": ""check_loan_status"", ""arguments"": {""application_id"": ""LA-1001""}}"
15:04:37,loan_service,tool_result,"{""name"": ""check_loan_status"", ""result"": {""found"": true, ""application_id"": ""LA-1001"", ""loan_type"": ""Home Loan"","
15:04:39,loan_service,llm_call,"{""step"": 4, ""model"": ""openai:gpt-4o-mini"", ""tool_calls"": [], ""tokens"": 1479}"
15:04:39,loan_service,final,"""Your loan application LA-1001 is currently **UNDER REVIEW**. It is in the credit assessment stage. The applic"


## 🚀 Live demo
```bash
streamlit run app.py      # chat + live audit-trail panel
python cli.py --demo      # scripted scenarios in the terminal
```

## ✅ Key messages
- **LLM app → tool → agent → multi-agent** is a small number of steps. The loop is ~30 lines.
- In BFSI, the value is in the **harness**: allow-lists, PII masking, no-decision rules, escalation, audit.
- **Skills** move business rules out of code and into reviewable playbooks.